# Frozen-magnitude coefficient refinement

This notebook runs one controlled A/B study, not a new WFR reproduction. Keep the current magnitude transport and old baseline predictions unchanged. Train the existing coefficient head with product supervision, then compare it with an otherwise identical arm that adds dose-profile supervision.

The primary checkpoint rule can retain the parent if no new state qualifies. That is an honest experimental outcome. The fixed-budget last states remain visible as secondary results.

Use one active runtime for this experiment. Follow R0–R10 in order on the first run. After reconnecting, restore R0/R1 and the needed dependencies, then resume the unfinished stage. The source, notebook, and documentation are self-contained; no extra helper upload is required.

Read `docs/PROTOCOL.md` before R3. Defaults are research settings, not optimized or clinical thresholds.

## R0 — Restore the exact code
This cell does not mount Drive, load checkpoints, or open medical arrays.

In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import zipfile

PAYLOAD_SHA256 = "ac6e9508bbe1e6435d3bb598dab46f3063650dc0d8d5b7f39f2cd874739307dc"
PAYLOAD_BASE64 = "..."
raw = base64.b64decode(PAYLOAD_BASE64)
if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError("The embedded source payload has changed.")
CODE = Path("/content") / ("wfr_coefficient_refinement_" + PAYLOAD_SHA256[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for item in archive.infolist():
        relative = Path(item.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Unsafe archive entry.")
        target = CODE / relative
        if not target.resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError("Archive entry escapes the code directory.")
        content = archive.read(item.filename)
        if target.exists():
            if not target.is_file() or target.read_bytes() != content:
                raise RuntimeError("An existing source file differs: " + str(target))
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
print("CODE:", CODE)
print("Source restored. No data or models were opened.")


## R1 — Mount Drive and set paths
The parent experiment and existing cache are read-only inputs. All new files go to a separate root.

In [ ]:
from pathlib import Path
from google.colab import drive
import os
import sys
import subprocess
import datetime
from collections import deque

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
PARENT_WORK = Path("/content/drive/MyDrive/ct2dose_thesis/conditional_wfr_dose")
CACHE = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1/cache.local/train192_seed42")
WORK = Path("/content/drive/MyDrive/ct2dose_thesis/wfr_coefficient_refinement")
HJD_EVALUATION = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1/evaluations.local/pilot_seed17_hjd_rf_eval01")
DIRECT_EVALUATION = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1/evaluations.local/pilot_seed17_residual_rf_eval01")
COMPOSITION_RUN = WORK / "runs.local/composition_seed29"
PROFILE_RUN = WORK / "runs.local/profile_seed29"
EVALUATION = WORK / "evaluations.local/locked_pair_val600"
CONFIG = CODE / "configs/paired.json"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "logs.local").mkdir(exist_ok=True)

os.environ.update({
    "POT_BACKEND_DISABLE_JAX": "1",
    "POT_BACKEND_DISABLE_TENSORFLOW": "1",
    "POT_BACKEND_DISABLE_CUPY": "1",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "JAX_PLATFORMS": "cpu",
    "CWFR_CPU_THREADS": "2",
})
import torch
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

def execute(arguments, cpu=False, stream=True):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(CODE) + os.pathsep + str(CODE / "parent_source")
    if cpu:
        env["CUDA_VISIBLE_DEVICES"] = ""
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    log = WORK / "logs.local" / (stamp + ".log")
    command = [sys.executable, "-u"] + [str(x) for x in arguments]
    print("RUN:", " ".join(command), flush=True)
    tail = deque(maxlen=18)
    with log.open("x", encoding="utf-8") as output:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=CODE, start_new_session=True)
        try:
            for line in proc.stdout:
                output.write(line); output.flush(); tail.append(line)
                if stream:
                    print(line, end="", flush=True)
            result = proc.wait()
        except BaseException:
            import signal
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()
            print("Stopped subprocess. Only sealed records/checkpoints are recoverable.")
            raise
    if not stream:
        print("".join(tail))
    print("LOG:", log)
    if result:
        raise RuntimeError("Command failed. Keep the log; do not bypass the error: " + str(log))
    return log

def task(*arguments, **kwargs):
    return execute(["-m", "refine", *arguments], **kwargs)

print("NEW WORK:", WORK)
print("READ-ONLY PARENT:", PARENT_WORK)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("No medical arrays have been opened.")


## R2 — Dependencies and CPU tests
Keep the working PyTorch build. Test tables use manufactured fixtures, not medical results. The full log is saved.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "tabulate": "tabulate"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Use a runtime with a working PyTorch installation; this notebook does not replace CUDA/PyTorch.")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
RUN_SOFTWARE_TESTS = True
if RUN_SOFTWARE_TESTS:
    execute(["-m", "unittest", "discover", "-s", CODE / "tests", "-v"], cpu=True, stream=False)
else:
    print("Tests not rerun. This does not waive source or environment checks.")


## R3 — Review and register the prospective protocol
Read the displayed settings before running the command. The1% relative guard is a development allowance, not an exact no-change promise or clinical criterion. Register once; do not edit the saved protocol after viewing results.

In [ ]:
import json
print((CODE / "docs/PROTOCOL.md").read_text(encoding="utf-8"))
print("CONFIGURATION:")
print(CONFIG.read_text(encoding="utf-8"))
task("register", "--cache", CACHE, "--parent-work", PARENT_WORK,
     "--work", WORK, "--config", CONFIG)


## R4 — Cache the frozen magnitude

Set `RUN_PREPARE=True` **in this cell**. Training magnitudes are inferred once from the parent; monitor40 magnitudes are reused. With `RUN_TO_COMPLETION=True`, the notebook automatically continues after each safe chunk. No optimizer is used.

If you must continue a partial cache under changed hardware, read the recovery document before setting `ALLOW_CACHE_ENVIRONMENT_CHANGE=True`.

In [ ]:
RUN_PREPARE = False
RUN_TO_COMPLETION = True
MAX_RECORDS_PER_CALL = 32
ALLOW_CACHE_ENVIRONMENT_CHANGE = False

if RUN_PREPARE:
    while True:
        arguments = ["prepare", "--cache", CACHE, "--parent-work", PARENT_WORK,
                     "--work", WORK, "--device", DEVICE, "--max-records", MAX_RECORDS_PER_CALL]
        if ALLOW_CACHE_ENVIRONMENT_CHANGE:
            arguments.append("--allow-cache-environment-change")
        task(*arguments)
        if (WORK / "frozen.local/COMPLETE.json").is_file() or not RUN_TO_COMPLETION:
            break
    marker = WORK / "frozen.local/COMPLETE.json"
    print(marker.read_text(encoding="utf-8") if marker.is_file() else "Cache paused; rerun this cell.")
else:
    print("NOT RUN: review the protocol, then set RUN_PREPARE=True in this cell.")


## R5 — Train the composition-supervised control
It starts from the selected parent coefficient, not from a new random head. A failed eligibility check is not a code error.

In [ ]:
RUN_COMPOSITION_ARM = False
RUN_TO_COMPLETION = True
MAX_UPDATES_PER_CALL = 128

if RUN_COMPOSITION_ARM:
    while True:
        task("train", "--work", WORK, "--arm", "composition", "--run", COMPOSITION_RUN,
             "--device", DEVICE, "--max-updates", MAX_UPDATES_PER_CALL)
        if (COMPOSITION_RUN / "COMPLETE.json").is_file() or not RUN_TO_COMPLETION:
            break
    summary = COMPOSITION_RUN / "training_summary.json"
    print(summary.read_text(encoding="utf-8") if summary.is_file() else "Control paused; rerun R5.")
else:
    print("NOT RUN: set RUN_COMPOSITION_ARM=True in this cell after the frozen cache is complete.")


## R6 — Train the profile-supervised arm
This arm uses the same parent initialization and batch order as R5. It does not start from the trained R5 model.

In [ ]:
RUN_PROFILE_ARM = False
RUN_TO_COMPLETION = True
MAX_UPDATES_PER_CALL = 128

if RUN_PROFILE_ARM:
    while True:
        task("train", "--work", WORK, "--arm", "profile", "--run", PROFILE_RUN,
             "--device", DEVICE, "--max-updates", MAX_UPDATES_PER_CALL)
        if (PROFILE_RUN / "COMPLETE.json").is_file() or not RUN_TO_COMPLETION:
            break
    summary = PROFILE_RUN / "training_summary.json"
    print(summary.read_text(encoding="utf-8") if summary.is_file() else "Profile arm paused; rerun R6.")
else:
    print("NOT RUN: set RUN_PROFILE_ARM=True in this cell to run the matched profile arm.")


## R7 — Review both completed arms
Keep the rules unchanged. Parent retention means no eligible new checkpoint was found. Both fixed-budget last states will remain visible in the secondary comparison.

In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for arm, folder in [("composition", COMPOSITION_RUN), ("profile", PROFILE_RUN)]:
    if not (folder / "COMPLETE.json").is_file():
        raise RuntimeError("Complete the declared budget first: " + str(folder))
    s = json.loads((folder / "training_summary.json").read_text(encoding="utf-8"))
    rows.append({"arm": arm, "status": s["status"],
                 "selected_extra_step": s["selected_additional_step"],
                 "extra_updates_executed": s["additional_updates_executed"],
                 **s["selected_monitor"]})
    print("\n", arm)
    display(pd.read_csv(folder / "monitor_history.csv"))
display(pd.DataFrame(rows))
print("This review uses monitor40, not a new validation600 evaluation.")


## R8 — Lock selections and evaluate the saved pair
Set the flag after reviewing R7. Selected and fixed-budget last states are evaluated. No WFR trajectory is recomputed. Validation600 remains a reused development cohort.

In [ ]:
RUN_LOCKED_EVALUATION = False
RUN_TO_COMPLETION = True
MAX_RECORDS_PER_CALL = 100

if RUN_LOCKED_EVALUATION:
    task("finalize", "--work", WORK, "--composition-run", COMPOSITION_RUN,
         "--profile-run", PROFILE_RUN)
    while True:
        task("evaluate", "--cache", CACHE, "--parent-work", PARENT_WORK,
             "--work", WORK, "--device", DEVICE, "--out", EVALUATION,
             "--max-records", MAX_RECORDS_PER_CALL)
        if (EVALUATION / "COMPLETE.json").is_file() or not RUN_TO_COMPLETION:
            break
else:
    print("NOT RUN: review both arms, then set RUN_LOCKED_EVALUATION=True in this cell.")


## R9 — Paired comparison with the saved old systems
The old HJD, direct residual flow and Practical final outputs are read, not retrained. A missing old file is an error to resolve, not a reason to invent a replacement.

In [ ]:
RUN_COMPARISON = False
COMPARISON = WORK / "comparisons.local/locked_pair"
if RUN_COMPARISON:
    task("compare", "--cache", CACHE, "--parent-work", PARENT_WORK, "--work", WORK,
         "--evaluation", EVALUATION, "--hjd-evaluation", HJD_EVALUATION,
         "--direct-evaluation", DIRECT_EVALUATION, "--out", COMPARISON)
else:
    print("NOT RUN: set RUN_COMPARISON=True after R8 completes.")


## R10 — Read the results
Read the case differences as well as the mean. Do not replace the primary selected result with a better-looking secondary last-state result.

In [ ]:
from IPython.display import display, Markdown, Image
import pandas as pd

if not (COMPARISON / "COMPLETE.json").is_file():
    raise RuntimeError("Complete R9 before opening the final report.")
with pd.option_context("display.precision", 10):
    display(pd.read_csv(COMPARISON / "comparison.csv"))
    display(pd.read_csv(COMPARISON / "fixed_budget_last.csv"))
    display(pd.read_csv(COMPARISON / "compute_scope.csv"))
    display(pd.read_csv(COMPARISON / "paired_case_deltas.private.csv", dtype={"case_id": str}))
display(Markdown((COMPARISON / "report.md").read_text(encoding="utf-8")))
for image in sorted((COMPARISON / "figures").glob("*.png")):
    display(Image(filename=str(image)))
print("REPORT:", COMPARISON / "report.md")
print("Keep arrays, checkpoints, identifiers and private provenance out of public uploads.")
